# Import, Clean & Format Existing Data

This notebook demonstrates how to work with your own data files using `dataset-builder`.

We'll walk through the full workflow:
1. **Create** sample CSV and JSON data files
2. **Import** them into the tool's internal format
3. **Clean** the data (deduplicate, trim whitespace, remove empty records)
4. **Format** for different ML frameworks (Alpaca, ChatML, ShareGPT)
5. **Export** the final dataset

This is the typical path when you already have raw data and need to prepare it for model training.

In [ ]:
import csv
import json
import os
from pathlib import Path

output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)

# --- Create sample CSV ---
csv_rows = [
    {"question": "What is Python?", "answer": "Python is a high-level, interpreted programming language known for its readability.", "category": "programming"},
    {"question": "What is machine learning?", "answer": "Machine learning is a subset of AI that enables systems to learn from data.", "category": "ai"},
    {"question": "What is a neural network?", "answer": "A neural network is a computing system inspired by biological neural networks.", "category": "ai"},
    {"question": "What is Git?", "answer": "Git is a distributed version control system for tracking changes in source code.", "category": "tools"},
    {"question": "What is Docker?", "answer": "Docker is a platform for building, shipping, and running applications in containers.", "category": "tools"},
    {"question": "What is an API?", "answer": "An API is an Application Programming Interface that allows software components to communicate.", "category": "programming"},
    {"question": "What is SQL?", "answer": "SQL is a domain-specific language used for managing relational databases.", "category": "data"},
    {"question": "What is a transformer model?", "answer": "A transformer is a deep learning architecture that uses self-attention mechanisms.", "category": "ai"},
    {"question": "What is REST?", "answer": "REST is an architectural style for designing networked applications using HTTP.", "category": "programming"},
    {"question": "What is fine-tuning?", "answer": "Fine-tuning is the process of adapting a pre-trained model to a specific task.", "category": "ai"}
]

csv_path = output_dir / "nb_sample.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "answer", "category"])
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"Created CSV with {len(csv_rows)} rows: {csv_path}")

# --- Create sample JSON with duplicates and empty fields ---
json_records = [
    {"question": "What is Python?", "answer": "Python is a high-level, interpreted programming language known for its readability.", "category": "programming"},
    {"question": "What is machine learning?", "answer": "Machine learning is a subset of AI that enables systems to learn from data.", "category": "ai"},
    {"question": "What is a neural network?", "answer": "A neural network is a computing system inspired by biological neural networks.", "category": "ai"},
    {"question": "What is Python?", "answer": "Python is a high-level, interpreted programming language known for its readability.", "category": "programming"},
    {"question": "", "answer": "", "category": ""},
    {"question": "What is Git?", "answer": "Git is a distributed version control system for tracking changes in source code.", "category": "tools"},
    {"question": "  What is Docker?  ", "answer": "  Docker is a platform for containers.  ", "category": "  tools  "},
    {"question": "What is machine learning?", "answer": "Machine learning is a subset of AI that enables systems to learn from data.", "category": "ai"},
    {"question": None, "answer": None, "category": None},
    {"question": "What is SQL?", "answer": "SQL is a domain-specific language used for managing relational databases.", "category": "data"},
    {"question": "What is an API?", "answer": "An API is an Application Programming Interface that allows software components to communicate.", "category": "programming"},
    {"question": "What is Python?", "answer": "Python is a high-level, interpreted programming language known for its readability.", "category": "programming"}
]

json_path = output_dir / "nb_sample.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_records, f, indent=2)

print(f"Created JSON with {len(json_records)} records (includes duplicates & empty): {json_path}")

In [ ]:
!npm start -- import -f ../output/nb_sample.csv -o ../output/nb_imported_csv.json

In [ ]:
!npm start -- import -f ../output/nb_sample.json -o ../output/nb_imported_json.json

In [ ]:
import json
from pathlib import Path

csv_imported = json.loads(Path("../output/nb_imported_csv.json").read_text(encoding="utf-8"))
json_imported = json.loads(Path("../output/nb_imported_json.json").read_text(encoding="utf-8"))

print(f"CSV imported records:  {len(csv_imported)}")
print(f"JSON imported records: {len(json_imported)}")
print()
print("First CSV record:")
print(json.dumps(csv_imported[0], indent=2))
print()
print("First JSON record:")
print(json.dumps(json_imported[0], indent=2))

## Cleaning Data

The `clean` command provides several cleaning operations:

- **`--dedupe`** — Remove duplicate records based on content similarity
- **`--trim`** — Strip leading/trailing whitespace from all text fields
- **`--remove-empty`** — Drop records where key fields are empty or null

We'll clean the JSON import which intentionally contains duplicates, empty records, and untrimmed text.

In [ ]:
!npm start -- clean -i ../output/nb_imported_json.json --dedupe --trim --remove-empty -o ../output/nb_cleaned.json -y

In [ ]:
import json
from pathlib import Path

before = json.loads(Path("../output/nb_imported_json.json").read_text(encoding="utf-8"))
after = json.loads(Path("../output/nb_cleaned.json").read_text(encoding="utf-8"))

print(f"Before cleaning: {len(before)} records")
print(f"After cleaning:  {len(after)} records")
print(f"Removed:         {len(before) - len(after)} records")
print()
print("Cleaned records:")
for i, rec in enumerate(after):
    q = rec.get("question", "N/A")
    print(f"  {i+1}. {q}")

## Formatting for ML Frameworks

Different training frameworks expect data in different formats:

| Format | Description | Common Use |
|--------|-------------|------------|
| **Alpaca** | `{instruction, input, output}` tuples | Stanford Alpaca, LoRA fine-tuning |
| **ChatML** | `{messages: [{role, content}]}` format | OpenAI-compatible models |
| **ShareGPT** | `{conversations: [{from, value}]}` format | ShareGPT-style datasets, Vicuna |

Let's format our cleaned data into all three formats.

In [ ]:
!npm start -- format -i ../output/nb_cleaned.json -f alpaca -o ../output/nb_alpaca/

In [ ]:
!npm start -- format -i ../output/nb_cleaned.json -f chatml -o ../output/nb_chatml/

In [ ]:
!npm start -- format -i ../output/nb_cleaned.json -f sharegpt -o ../output/nb_sharegpt/

In [ ]:
!npm start -- export -i ../output/nb_cleaned.json -o ../output/nb_final.csv -f csv

In [ ]:
import csv
from pathlib import Path

csv_path = Path("../output/nb_final.csv")
if csv_path.exists():
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    print(f"Final CSV has {len(rows)} rows")
    print(f"Columns: {list(rows[0].keys()) if rows else 'N/A'}")
    print()
    for i, row in enumerate(rows[:3]):
        print(f"Row {i+1}: {dict(row)}")
    if len(rows) > 3:
        print(f"  ... and {len(rows) - 3} more rows")
else:
    print(f"File not found: {csv_path}")

In [ ]:
import shutil
from pathlib import Path

cleanup_files = [
    "../output/nb_sample.csv",
    "../output/nb_sample.json",
    "../output/nb_imported_csv.json",
    "../output/nb_imported_json.json",
    "../output/nb_cleaned.json",
    "../output/nb_final.csv",
]
cleanup_dirs = [
    "../output/nb_alpaca",
    "../output/nb_chatml",
    "../output/nb_sharegpt",
]

for f in cleanup_files:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Deleted: {f}")

for d in cleanup_dirs:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)
        print(f"Deleted: {d}")

print("\nCleanup complete.")